# Prueba de humo en Kabré

Ejecuta la celda de abajo (Shift+Enter) **en el nodo donde vas a entrenar**. Escribe un archivo `humo_<nodo>.json` (sin contraseñas ni claves) y lo imprime: nodo, GPU, memoria, disco, si hay internet y qué librerías están instaladas. Pega ese resultado a Claude.

Hazlo primero en una cola de pruebas (por ejemplo `kura-test` o `nukwa-debug`) y luego en la cola donde vayas a entrenar.

In [ ]:
import json, os, platform, shutil, socket, subprocess, sys, urllib.request

def run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=30).stdout.strip()
    except Exception as exc:
        return f"no disponible: {exc}"

def internet(url):
    try:
        req = urllib.request.Request(url, method="HEAD", headers={"User-Agent": "nano2predict-humo"})
        with urllib.request.urlopen(req, timeout=8) as r:
            return f"ok {r.status}"
    except Exception as exc:
        return f"sin acceso: {str(exc)[:80]}"

home = os.path.expanduser("~")
total, usado, libre = shutil.disk_usage(home)
info = {
    "nodo": socket.gethostname(),
    "python": sys.version.split()[0],
    "sistema": platform.platform(),
    "cpus": os.cpu_count(),
    "memoria_MB": next((int(l.split()[1]) // 1024 for l in open("/proc/meminfo") if l.startswith("MemTotal")), None) if os.path.exists("/proc/meminfo") else None,
    "disco_home_GB": {"total": round(total / 1e9, 1), "libre": round(libre / 1e9, 1)},
    "slurm_job": os.environ.get("SLURM_JOB_ID"),
    "slurm_particion": os.environ.get("SLURM_JOB_PARTITION"),
    "gpu_nvidia_smi": run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"]),
    "internet": {u: internet(u) for u in ("https://pypi.org", "https://github.com", "https://download.pytorch.org")},
    "modulos_ml": {},
}
for mod in ("numpy", "pandas", "sklearn", "torch", "torchvision", "timm", "PIL", "matplotlib"):
    try:
        m = __import__(mod)
        info["modulos_ml"][mod] = getattr(m, "__version__", "ok")
    except Exception:
        info["modulos_ml"][mod] = "no instalado"
try:
    import torch
    info["torch_cuda"] = bool(torch.cuda.is_available())
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        info["torch_gpu"] = {"nombre": p.name, "memoria_GB": round(p.total_memory / 1e9, 1)}
except Exception:
    pass
salida = f"humo_{info['nodo']}.json"
with open(salida, "w", encoding="utf-8") as f:
    json.dump(info, f, ensure_ascii=False, indent=2)
print(json.dumps(info, ensure_ascii=False, indent=2))
print("\nEscrito:", os.path.abspath(salida))